In [4]:
import boto3

region = "us-east-1"
endpoint_name = "flight-price-xgb-endpoint"  # pipeline parameter로 준 이름

sm = boto3.client("sagemaker", region_name=region)
print(sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"])


InService


In [5]:
import json, boto3

region = "us-east-1"
endpoint_name = "flight-price-xgb-endpoint"

rt = boto3.client("sagemaker-runtime", region_name=region)

payload = {
  # inference.py는 {"records": [...]} 형태도 지원
  "records": [
    {
      # 여기에 "전처리 후 피처"가 아니라,
      # inference.py가 요구하는 원본 feature 컬럼을 넣어야 합니다.
      # (preprocess에서 만든 ct.feature_names_in_ 기준)
      "purchase_day_of_week": 2,
      "purchase_time_bucket": "morning",
      "days_until_departure": 10,
      "days_until_departure_bucket": 1,
      "stops_count": 1,
      "flight_duration_bucket": "medium",
      "route_hash": 12345678,
      "prev_fare": 20000,
      "min_fare_last_7d": 18000,
      "mean_fare_last_7d": 21000,
      "min_fare_last_14d": 17500,
      "mean_fare_last_14d": 22000,
      "min_fare_last_30d": 17000,
      "mean_fare_last_30d": 23000,
      "prev_fare_vs_min_7d": 1.11,
      "prev_fare_vs_mean_7d": 0.95,
      "prev_fare_vs_min_14d": 1.14,
      "prev_fare_vs_mean_14d": 0.91,
      "prev_fare_vs_min_30d": 1.18,
      "prev_fare_vs_mean_30d": 0.87,
      "is_weekend_departure": 0,
      "is_holiday_season": 0
    }
  ]
}

resp = rt.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="text/csv",
    Body=json.dumps(payload).encode("utf-8"),
)
print(resp["Body"].read().decode("utf-8"))


9.607357025146484

